# 1-NN classification with leave one process out
We only focus on 1-NN and break the tails by introducing a random picking of the labels:
1. Compute the Euclidean distance between the feature computed from the time series on the training dataset and the tests
2. Take the minimum distance
3. Extract the indeces of the time series that have the minimum distance
4. Pick a label that is random among them

In [ ]:
import numpy as np
import pandas as pd
import os
import tqdm

In [ ]:
models_dsct=['GNO', 'UNO', 'PINK', 'BROWN', 'VIOLET', # noises (R)
        'AR1_GNO', 'STAR_GNO', #    ARMA (R)
        'ARNOLD', 'CHIRIKOV', # Conservative chaotic maps (R)
        'HENR_diverse', 'HENR_same', 'QUADRATIC_RSUM', # Sum of frwd and bckwd realisations of chaotic maps (R)
        'AR1_UNO', 'ARMA11_UNO', 'AR3_Gamma', 'N_AR2', 'SETAR1_GNO', 'SETAR2_GNO',
        'HEN', 'HEN_SUM', 'LOGISTIC4', 'LOGISTIC38284', 'QUADRATIC', # Chaotic maps (I)
        'MODA', 'LLOG', # Other deterministic (I)
        'SINE_STOCH' # Other stochastic (I)
        ]



models_cnt = ['BRW_cont','OU', 'Oscillator', 
              'LORENZ_SUM', 'ROSSLER_SUM', 
              'MACKEYGLASS17', 'VDP', 
              'LORENZ_STOCH_SUM',
                'VDP_STOCH']


models = models_dsct + models_cnt
print('Models:', models)

model_keywords={ # discrete
                'GNO': 'reversible', 'UNO': 'reversible', 'PINK': 'reversible', 'BROWN': 'reversible', 'VIOLET': 'reversible',
                'AR1_GNO': 'reversible', 'STAR_GNO': 'reversible',
                'ARNOLD': 'reversible', 'CHIRIKOV': 'reversible',
                'HENR_diverse': 'reversible', 'HENR_same': 'reversible', 'QUADRATIC_RSUM': 'reversible',
                'AR1_UNO': 'irreversible', 'ARMA11_UNO': 'irreversible', 'AR3_Gamma': 'irreversible', 'N_AR2': 'irreversible', 'SETAR1_GNO': 'irreversible', 'SETAR2_GNO': 'irreversible', 
                'HEN': 'irreversible', 'HEN_SUM': 'irreversible', 'LOGISTIC4': 'irreversible', 'LOGISTIC38284': 'irreversible','QUADRATIC': 'irreversible',
                'MODA': 'irreversible', 'LLOG': 'irreversible',
                'SINE_STOCH': 'irreversible',
                # continuous
                'BRW_cont': 'reversible', 'OU': 'reversible', 'Oscillator': 'reversible',
                'LORENZ_SUM': 'irreversible', 'ROSSLER_SUM': 'irreversible',
                'MACKEYGLASS17': 'irreversible', 'VDP': 'irreversible',
                'LORENZ_STOCH_SUM': 'irreversible',
                'VDP_STOCH': 'irreversible'
        }

In [ ]:
num_ts=100
models_repeated = np.repeat(models, num_ts)

In [ ]:
cwd = os.getcwd()

dir_hctsa= cwd+'/../../../data-tr/main-analysis/data-analysis/'

dir_accuracy= cwd+'/data-analysis/accuracy/'

os.makedirs(dir_accuracy, exist_ok=True)

In [ ]:
df_hctsa=pd.read_csv(dir_hctsa+'df_TS_DataMat_diff.csv')
df_hctsa.set_index(['Model'], inplace=True)

# Add column with reversible/irreversible label
df_type=df_hctsa.index.get_level_values(0).map(model_keywords)
df_hctsa.insert(0,'Type',df_type)

In [ ]:
def NearestNeighbor_random(x_train, x_test, y_train):
    """
    1-NN classifier with Euclidean distance and randomized tie-breaking.

    Parameters:
    - x_train: (n_train, n_features) numpy array
    - x_test:  (n_test, n_features) numpy array
    - y_train: (n_train,) numpy array of labels

    Returns:
    - predicted_labels: (n_test,) numpy array of predicted labels
    """

    # Compute pairwise Euclidean distances between x_test and x_train
    # Shape: (n_test, n_train)
    distances = np.linalg.norm(x_test[:, None, :] - x_train[None, :, :], axis=2)

    # For each test sample, find the index(es) of the nearest training sample(s)
    min_values = np.min(distances, axis=1)  # shape: (n_test,)
    
    # Set seed for reproducibility
    np.random.seed(42)
    # Randomized tie-breaking for each row
    nearest_indices = [
        np.random.choice(np.where(distances[i] == min_values[i])[0])
        for i in range(distances.shape[0])
    ]

    predicted_labels = y_train[nearest_indices]
    return predicted_labels


In [ ]:
# List of operations
ops=df_hctsa.columns[1:].values

dict_ypred={}
dict_accuracy={}
dict_true_false={}
for op in tqdm.tqdm(ops):
    y_pred = [] # List to store the predicted labels
    for model in models:
        # Leave one process out: test is the model out, train are the others 
        df_model=df_hctsa.loc[ model,['Type', op]]
        df_other=df_hctsa.loc[ df_hctsa.index.get_level_values(0)!=model,['Type', op]]

        # Separate data from labels
        X_train=np.abs(df_other[op].values)
        y_train=df_other['Type'].values

        X_test=np.abs(df_model[op].values)
        y_test=df_model['Type'].values

        # Reshape the data
        X_train = X_train.reshape(-1, 1)
        X_test = X_test.reshape(-1, 1)

        # Nearest Neighbor
        y_pred_model = NearestNeighbor_random(X_train, X_test, y_train)
        y_pred.append(y_pred_model)
    
    y_pred=np.array(y_pred).flatten()   
    dict_ypred[op]=y_pred
    # Populate True/False dictionary    
    dict_true_false[op]= y_pred == df_hctsa.loc[:,'Type'].values
    # Compute the accuracy
    accuracy = np.mean(dict_true_false[op])

    dict_accuracy[op]=accuracy


In [ ]:
## Save the predictions
# Convert into a DataFrame
df_pred=pd.DataFrame.from_dict(dict_ypred, orient='columns')
df_pred['Model'] = models_repeated
df_pred = df_pred.set_index('Model')
df_pred.to_csv(dir_accuracy+'df_pred_abs_1NN.csv')

## Save the True/False
# Convert into a DataFrame
df_true_false=pd.DataFrame.from_dict(dict_true_false, orient='columns')
df_true_false['Model'] = models_repeated
df_true_false = df_true_false.set_index('Model')
df_true_false.to_csv(dir_accuracy+'df_true_false_abs_1NN.csv')
# Save the accuracy
# Convert into a DataFrame
df_accuracy=pd.DataFrame.from_dict(dict_accuracy, orient='index', columns=['Accuracy'])
df_accuracy.index.name='Operation'
df_accuracy.to_csv(dir_accuracy+'df_accuracy_abs_1NN.csv')

# Sort the operations by accuracy
df_accuracy.sort_values(by='Accuracy', ascending=False, inplace=True)
# Save the sorted DataFrame
df_accuracy.to_csv(dir_accuracy+'df_accuracy_abs_1NN_sorted.csv')